In [ ]:
import pandas as pd

df = pd.read_csv('/content/telco-churn-api/data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.info()

In [ ]:
print((df['TotalCharges'] == " ").sum())

In [ ]:
from numpy import float64
df['TotalCharges'] = df['TotalCharges'].replace(" ",0).astype(float)

In [ ]:
df['TotalCharges'].info()

In [ ]:
df.drop(['customerID'], axis=1, inplace=True)

In [ ]:
df.info()

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

In [ ]:
df['Churn'].value_counts()

In [ ]:
yes_no_cols = ['Partner', 'Dependents', 'PhoneService','PaperlessBilling']
df[yes_no_cols] = df[yes_no_cols].replace({'Yes':1, 'No':0})

In [ ]:
cat_cols = df.select_dtypes(include='object')
cat_cols = cat_cols.columns.tolist()

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, dtype=int, drop_first=True)

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())

In [ ]:
from xgboost import XGBClassifier

neg, pos = 4139,1495
scale = neg/pos

model = XGBClassifier(
    scale_pos_weight = scale,
    random_state=42,
    eval_metric = 'logloss'
)

model.fit(X_train, y_train)
print('The model has been trained!')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print('\nConfusion Matrix')
print(confusion_matrix(y_test, y_pred))
print('\nReport')
print(classification_report(y_test, y_pred))

In [ ]:
feat_imp = pd.DataFrame({'feature':X.columns, 'importance':model.feature_importances_}).sort_values(by='importance',ascending=False)

In [ ]:
feat_imp

In [ ]:
y_proba = model.predict_proba(X_test)[:,1]
y_pred_03 = (y_proba >= 0.3).astype(int)

In [ ]:
print(classification_report(y_test, y_pred_03))
print(confusion_matrix(y_test, y_pred_03))

In [ ]:
print(df.groupby('Contract_Two year')['Churn'].mean())
print(df.groupby('InternetService_Fiber optic')['Churn'].mean())

In [ ]:
import numpy as np
test_df = pd.read_csv('/content/telco-churn-api/Download-10-New-Customers-CSV.csv')

X_new_raw = test_df.drop('customerID', axis=1)
X_new_encoded = pd.get_dummies(X_new_raw)
X_new_final = X_new_encoded.reindex(columns= X.columns, fill_value=0)

proba = model.predict_proba(X_new_final)[:,1]
test_df['Churn Chance'] = np.round(proba*100, 1)
test_df['Prediction'] = ['About to go' if p >=0.3 else 'Loyal' for p in proba]

print(test_df[['customerID','Contract','InternetService','tenure','Churn Chance','Prediction']])